# 04 Fine-tuning on our own data
Pre-trained UCI CNN -> freeze all but the last two layers -> Adam(1e-4) -> train on p1-p3 -> test on p4, p5.

Three numbers on our test people (p4, p5):
1. UCI model, no fine-tuning
2. Fine-tuned model
3. Model trained only on our data

Early stopping uses the last training person (p3) as validation. Data: processed CSVs in `OWN_ROOT` (default `../data/own/processed`, copied from Drive `HAR-data/own/processed/`).

In [ ]:
import os, sys, yaml, numpy as np, pandas as pd
sys.path.insert(0, "../src")
from tensorflow import keras
from data_loader import load_own
from preprocess import normalize
from models import build_cnn
from train import fit, fine_tune
from evaluate import report, plot_learning_curves
cfg = yaml.safe_load(open("../config/config.yaml"))
OWN_ROOT = os.environ.get("OWN_ROOT", "../data/own/processed")
FIG = "../reports/figures"
own = load_own(OWN_ROOT, cfg)
X = normalize(own["X"], "../" + cfg["norm_stats"])           # same UCI train stats the app uses
tr_people, te_people = cfg["split"]["own_train"], cfg["split"]["own_test"]
val_person = tr_people[-1]
m_tr = np.isin(own["subject"], tr_people[:-1]); m_va = own["subject"] == val_person
m_te = np.isin(own["subject"], te_people)
print({p: int((own["subject"] == p).sum()) for p in sorted(set(own["subject"]))}, "windows per person")

In [ ]:
uci = keras.models.load_model("../models/cnn_uci.keras")
r_uci = report(own["y"][m_te], uci.predict(X[m_te], verbose=0).argmax(1), FIG, "own_uci_only")

ft, h_ft = fine_tune(uci, X[m_tr], own["y"][m_tr], X[m_va], own["y"][m_va], cfg)
plot_learning_curves(h_ft, f"{FIG}/learning_curves_finetune.png")
ft.save("../models/cnn_finetuned.keras")
r_ft = report(own["y"][m_te], ft.predict(X[m_te], verbose=0).argmax(1), FIG, "own_finetuned")

own_only, _ = fit(build_cnn(), X[m_tr], own["y"][m_tr], X[m_va], own["y"][m_va], cfg)
r_own = report(own["y"][m_te], own_only.predict(X[m_te], verbose=0).argmax(1), FIG, "own_only")

In [ ]:
pd.DataFrame({k: {"accuracy": r["accuracy"], "macro_f1": r["macro_f1"]}
              for k, r in [("UCI, no fine-tuning", r_uci), ("Fine-tuned", r_ft), ("Own data only", r_own)]}).T.round(3)